[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-2/blob/main/notebooks/04_summary.ipynb)

# Сводная проверка

Короткая проверка, что графический метод и симплекс-метод дают одинаковые ответы.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

c = np.array([-3.0, 6.0])
constraints = [
    (np.array([5.0, -2.0]), "<=", 4.0, "5*x1 - 2*x2 <= 4"),
    (np.array([1.0, -2.0]), ">=", -4.0, "x1 - 2*x2 >= -4"),
    (np.array([1.0, 1.0]), ">=", 4.0, "x1 + x2 >= 4"),
]

def objective(point):
    return float(c @ point)

def feasible(point, tol=1e-9):
    if point[0] < -tol or point[1] < -tol:
        return False
    for a, sense, b, _ in constraints:
        left = float(a @ point)
        if sense == "<=" and left > b + tol:
            return False
        if sense == ">=" and left < b - tol:
            return False
    return True

def active(point, tol=1e-7):
    names = []
    if abs(point[0]) <= tol:
        names.append("x1 = 0")
    if abs(point[1]) <= tol:
        names.append("x2 = 0")
    for a, _, b, label in constraints:
        if abs(float(a @ point) - b) <= tol:
            names.append(label)
    return "; ".join(names)

from itertools import combinations

boundaries = constraints + [
    (np.array([1.0, 0.0]), ">=", 0.0, "x1 >= 0"),
    (np.array([0.0, 1.0]), ">=", 0.0, "x2 >= 0"),
]

points = []
for first, second in combinations(boundaries, 2):
    a1, _, b1, _ = first
    a2, _, b2, _ = second
    matrix = np.vstack([a1, a2])
    if abs(np.linalg.det(matrix)) < 1e-9:
        continue
    point = np.linalg.solve(matrix, np.array([b1, b2]))
    if feasible(point):
        if not any(np.linalg.norm(point - old, ord=np.inf) < 1e-7 for old in points):
            points.append(point)

rows = []
for index, point in enumerate(sorted(points, key=lambda p: (p[0], p[1])), start=1):
    rows.append({
        "point": f"A{index}",
        "x1": point[0],
        "x2": point[1],
        "Z": objective(point),
        "active constraints": active(point),
    })

vertices = pd.DataFrame(rows)
vertices

In [ ]:
min_row = vertices.loc[vertices['Z'].idxmin()]
max_rows = vertices[abs(vertices['Z'] - vertices['Z'].max()) < 1e-7]
summary = pd.DataFrame([
    {'task': 'minimum', 'point': f"({min_row.x1:.6g}; {min_row.x2:.6g})", 'Z': min_row.Z},
    {'task': 'maximum edge start', 'point': f"({max_rows.iloc[0].x1:.6g}; {max_rows.iloc[0].x2:.6g})", 'Z': max_rows.iloc[0].Z},
    {'task': 'maximum edge end', 'point': f"({max_rows.iloc[-1].x1:.6g}; {max_rows.iloc[-1].x2:.6g})", 'Z': max_rows.iloc[-1].Z},
])
summary

Для максимума получается не одна точка, а отрезок. Это нормально: линии уровня целевой функции параллельны одной стороне допустимого треугольника.